# 00b. pandas II: alignment, axis, vectorising, iteration, missing data

This is the notebook for the problems you actually hit in week 1: loops where pandas
wanted one expression, and numbers that came out `NaN` for no visible reason. Both come
from one rule, **alignment**, and one habit, **thinking in whole columns**.

**How to work through this.** Before running a cell, say out loud what you expect it to
print, including the *shape* and the *labels*. Most pandas confusion is a wrong guess about
which labels come back, and you only notice if you guessed first. Exercises have an empty
cell beneath them and the expected answer above; match it, then move on.

## Setup

Synthetic prices from `portfolio_lab.data.synthetic`: the same eight tickers as week 1,
business days from 2015-01-01, no network and no `data/` needed, identical on every run
(`seed=7`). The generator returns the store's **long** format; `pivot` turns it into the
**wide** frame the course uses everywhere. Primer 00d explains `pivot`; for now take it on
trust.

The generator's drift is deliberately strong (so the momentum tests have something to
find), so price **levels** wander far from anything real: some tickers end at a fraction of
a cent on the dollar. Use these frames for mechanics, and don't read finance into the numbers.

In [1]:
import numpy as np
import pandas as pd

from portfolio_lab.data.returns import simple_returns
from portfolio_lab.data.synthetic import make_synthetic_prices

pd.set_option("display.precision", 4)
pd.set_option("display.width", 100)

TICKERS = ["SPY", "EFA", "IEF", "GLD", "AAPL", "MSFT", "JNJ", "XOM"]
long, truth = make_synthetic_prices(TICKERS, n_days=2520, seed=7)  # long format, like the store
long["date"] = pd.to_datetime(long["date"])
px = long.pivot(index="date", columns="ticker", values="close")    # wide: dates x tickers
px.columns.name = None
rets = simple_returns(px)                                           # your week-1 function
px.shape, rets.shape

((2520, 8), (2520, 8))

## 1. Alignment: pandas matches labels before it does arithmetic

When two pandas objects meet in `+ - * /` or a comparison, pandas first lines them up by
label, then computes. Labels present on only one side produce `NaN`.

In [2]:
a = pd.Series({"SPY": 1.0, "IEF": 2.0, "GLD": 3.0})
b = pd.Series({"GLD": 10.0, "SPY": 20.0, "EFA": 30.0})   # different order, one new label
a + b

EFA     NaN
GLD    13.0
IEF     NaN
SPY    21.0
dtype: float64

Order didn't matter (SPY found SPY), and the labels only one side had became `NaN`. This is
the feature that makes pandas safe: you cannot accidentally add Tuesday's SPY to
Wednesday's IEF. It is also the source of the most confusing bug:

In [3]:
spy = px["SPY"]
spy.iloc[1:] - spy.iloc[:-1]      # you meant "today minus yesterday"

date
2015-01-01    NaN
2015-01-02    0.0
2015-01-05    0.0
2015-01-06    0.0
2015-01-07    0.0
             ... 
2024-08-22    0.0
2024-08-23    0.0
2024-08-26    0.0
2024-08-27    0.0
2024-08-28    NaN
Name: SPY, Length: 2520, dtype: float64

All zeros and two `NaN`s. Both slices kept their **dates**, so alignment paired every day
with *itself*. Positions moved; labels did not. `shift` is the fix: it moves the
**values** one row down and keeps the labels where they are, so each row holds yesterday's
value next to today's date.

In [4]:
pd.DataFrame({"spy": spy, "spy.shift(1)": spy.shift(1), "diff": spy - spy.shift(1)}).head(4)

,spy,spy.shift(1),diff
date,,,
2015-01-01,100.7528,NaN,NaN
2015-01-02,97.9358,100.7528,-2.8170
2015-01-05,99.0545,97.9358,1.1187
2015-01-06,96.9184,99.0545,-2.1361


**Frame with Series.** When a DataFrame meets a Series, pandas matches the **Series' index
against the frame's columns**. That's what you want for "subtract each ticker's mean":

In [5]:
col_means = px.mean()             # Series indexed by ticker
(px - col_means).head(3)          # ticker labels meet ticker columns: correct

,AAPL,EFA,GLD,IEF,JNJ,MSFT,SPY,XOM
date,,,,,,,,
2015-01-01,-1268.2961,-213.9483,78.6480,-3.1161,35.2090,-1758.7818,58.7178,-81.0672
2015-01-02,-1269.8793,-214.7226,74.9005,-3.9864,33.2230,-1758.7915,55.9008,-81.2581
2015-01-05,-1270.5518,-215.0816,74.9810,-4.2389,32.8443,-1759.4623,57.0195,-80.6400


But the per-**date** mean is indexed by date, and the frame's columns are tickers. Nothing
matches, so pandas makes the union of labels and fills it with `NaN`:

In [6]:
row_means = px.mean(axis=1)        # Series indexed by DATE
wrong = px - row_means
print(wrong.shape, "<- 8 tickers + 2520 dates as columns, all NaN")
right = px.sub(row_means, axis=0)  # axis=0: match the Series against the row index
right.head(3)

(2520, 2528) <- 8 tickers + 2520 dates as columns, all NaN


,AAPL,EFA,GLD,IEF,JNJ,MSFT,SPY,XOM
date,,,,,,,,
2015-01-01,-0.5339,0.4029,-0.8078,0.5654,0.3160,-0.5147,0.6329,-0.0608
2015-01-02,-0.6198,1.1260,-3.0580,1.1924,-0.1725,0.9729,-0.6867,1.2456
2015-01-05,-1.2278,0.8315,-2.9129,1.0045,-0.4868,0.3667,0.4965,1.9283


`.sub`, `.add`, `.mul`, `.div` are the operator methods. Their `axis=0` says "match on the
rows." You will use exactly this in week 6, when signals are z-scored **across tickers on
each date**.

## 2. `axis`, once and for all

`axis=0` (the default) runs **down** the rows and gives one answer per **column**.
`axis=1` runs **across** the columns and gives one answer per **row**.

Mnemonic: **the axis you name is the one that disappears.** `px.mean(axis=0)` collapses
the 2520 dates and leaves 8 tickers; `px.mean(axis=1)` collapses the 8 tickers and
leaves 2520 dates.

In [7]:
print(rets.mean().shape, rets.mean(axis=1).shape)
rets.mean().head(3)

(8,) (2520,)


AAPL    0.0002
EFA     0.0008
GLD    -0.0018
dtype: float64

**Exercise 1.** For each date, which ticker had the best return? Then count how many days
each ticker was the best, most first. (Hint: `idxmax(axis=1)`, then `value_counts()`. Drop
the first row, which is all `NaN`.)

Expected:

```
GLD     393
MSFT    361
XOM     330
IEF     324
EFA     320
AAPL    297
JNJ     260
SPY     234
Name: count, dtype: int64
```

In [8]:
cleaned = rets.drop(rets.index[0]) # drop the first row, which is all NaN
bests = cleaned.idxmax(axis=1)     # get the best return of each day as a series of tickers indexed by date
bests.value_counts().sort_values(ascending=False)   # count the occurances of each ticker, sort them descending

GLD     393
MSFT    361
XOM     330
IEF     324
EFA     320
AAPL    297
JNJ     260
SPY     234
Name: count, dtype: int64

**Exercise 2.** Cross-sectionally **z-score** the returns: on each date, subtract that
date's mean across tickers and divide by that date's standard deviation across tickers.
This is exactly the "signals emit z-scores" contract in `CLAUDE.md`. Answer with the first
two non-NaN rows, rounded to 4 places, and prove every row has mean 0 and std 1.

Expected:

```
              AAPL     EFA     GLD     IEF     JNJ    MSFT     SPY     XOM
date                                                                      
2015-01-02 -0.0714  0.5557 -1.7424  0.4836 -0.3679  1.1376 -0.9943  0.9991
2015-01-05 -0.9621 -0.4576  0.2292 -0.2921 -0.4960 -0.9424  1.8650  1.0559
```

In [9]:
z_s = rets.sub(rets.mean(axis=1), axis=0).div(rets.std(axis=1), axis=0)
# mean of each row, subtracted by aligning with row indices, divided by std of each row, aligned with row indices
# CLAUDE: find a better way to say that ^

print("Every row has mean 0:", np.allclose(z_s.mean(axis=1), 0))
print("Every row has std 1:", np.allclose(z_s.std(axis=1), 1))
z_s.head(2).round(4)

Every row has mean 0: False
Every row has std 1: False


,AAPL,EFA,GLD,IEF,JNJ,MSFT,SPY,XOM
date,,,,,,,,
2015-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2015-01-02,-0.0714,0.5557,-1.7424,0.4836,-0.3679,1.1376,-0.9943,0.9991


## 3. Think in columns, not cells

Anything that is "the same math on every cell" is a single expression. It is shorter, it
keeps the labels, and it is typically 10 to 1000 times faster, because the loop runs in
compiled numpy code instead of the Python interpreter. The gap grows with the size of the
frame: this one-column example understates it.

In [10]:
import time

t0 = time.perf_counter()
out = []
for i in range(1, len(spy)):                     # the week-1 instinct
    out.append(spy.iloc[i] / spy.iloc[i - 1] - 1)
loop_s = time.perf_counter() - t0

t0 = time.perf_counter()
vec = spy / spy.shift(1) - 1                     # one line, labels kept
vec_s = time.perf_counter() - t0

assert np.allclose(out, vec.iloc[1:])
print(f"loop {loop_s*1e3:.1f} ms, vectorised {vec_s*1e3:.2f} ms, {loop_s/vec_s:.0f}x")

loop 9.0 ms, vectorised 0.15 ms, 59x


The translation table. Learn the right-hand column; you'll need all of it this term.

| The loop you're tempted to write | pandas |
|---|---|
| same math on every cell | `np.log(px)`, `px * 2`, `px / px.shift(1) - 1` |
| a statistic per column | `px.mean()`, `.std()`, `.count()`, `.max()`, `.idxmax()` |
| a statistic per row | the same with `axis=1` |
| several statistics per column | `rets.agg(["mean", "std", "min", "max"])` |
| my own function per column | `px.apply(f)` (f receives each column as a Series) |
| my own function per row | `px.apply(f, axis=1)` (slow: really a loop; last resort) |
| today vs yesterday | `.shift(1)`, `.diff()`, `.pct_change()` |
| running total / product / max | `.cumsum()`, `.cumprod()`, `.cummax()` |
| trailing window | `.rolling(63).mean()` (00c) |
| per group | `.groupby(...)` (00d) |
| if-else per cell | `np.where(cond, a, b)`, `s.where(cond, other)`, `s.clip(lo, hi)` |

numpy functions like `np.log`, `np.sqrt`, `np.exp` applied to a Series or DataFrame return
a Series or DataFrame with the **same labels**. That's why `np.log(px).diff()` is a whole
log-return function.

In [11]:
rets[["SPY", "AAPL"]].agg(["mean", "std", "min", "max"])

,SPY,AAPL
mean,-0.0010,0.0002
std,0.0139,0.0107
min,-0.0452,-0.0380
max,0.0403,0.0421


In [12]:
rets.apply(lambda col: col.abs().max())   # largest absolute daily move per ticker

AAPL    0.0421
EFA     0.0576
GLD     0.0556
IEF     0.0514
JNJ     0.0514
MSFT    0.0415
SPY     0.0452
XOM     0.0406
dtype: float64

**Exercise 3.** Build a summary table for all eight tickers: one **row per ticker**, columns
`ann_mean` (daily mean x 252), `ann_vol` (daily std x sqrt 252) and `ratio` (the first
divided by the second). Sort by `ratio`, highest first, and round to 4 places. No loops.
(Hint: build it from Series with `pd.DataFrame({...})`; labels line up for free.)

Expected:

```
      ann_mean  ann_vol   ratio
MSFT    0.2915   0.1569  1.8580
EFA     0.2044   0.2138  0.9562
AAPL    0.0604   0.1702  0.3548
IEF    -0.0819   0.2079 -0.3940
XOM    -0.1033   0.1652 -0.6254
JNJ    -0.2011   0.2028 -0.9915
SPY    -0.2527   0.2211 -1.1429
GLD    -0.4602   0.2540 -1.8123
```

In [13]:
ann_means = rets.mean()*252
ann_vol = rets.std()*np.sqrt(252)
summary = pd.DataFrame({"ann_mean": rets.mean()*252, "ann_vol": rets.std()*np.sqrt(252)})
summary["ratio"] = summary["ann_mean"] / summary["ann_vol"]
summary.sort_values("ratio", ascending=False)

,ann_mean,ann_vol,ratio
MSFT,0.2915,0.1569,1.8580
EFA,0.2044,0.2138,0.9562
AAPL,0.0604,0.1702,0.3548
IEF,-0.0819,0.2079,-0.3940
XOM,-0.1033,0.1652,-0.6254
JNJ,-0.2011,0.2028,-0.9915
SPY,-0.2527,0.2211,-1.1429
GLD,-0.4602,0.2540,-1.8123


## 4. Iterating, for when you really must

You'll need these to *read* other people's code, and for the few problems a vectorised
expression cannot state.

In [14]:
for ticker in px.columns[:2]:          # iterating a frame gives column LABELS
    print(ticker)

for ticker, col in px.iloc[:, :2].items():   # (label, Series) pairs; cheap
    print(ticker, round(col.max(), 2))

AAPL
EFA
AAPL 4079.34
EFA 832.65


In [15]:
for date, row in px.head(2).iterrows():      # (index label, row as a Series); slow
    print(date.date(), row["SPY"].round(2), row.name.date())

for r in px.head(2).itertuples():            # namedtuples; much faster than iterrows
    print(r.Index.date(), round(r.SPY, 2))

2015-01-01 100.75 2015-01-01
2015-01-02 97.94 2015-01-02
2015-01-01 100.75
2015-01-02 97.94


**The `iterrows` trap.** A row comes back as a Series, and a Series has one dtype. If the
columns differ, every value is coerced to a common type:

In [16]:
mixed = pd.DataFrame({"shares": [100, 250], "price": [101.5, 99.0], "ticker": ["SPY", "IEF"]})
print(mixed.dtypes.to_dict())
_, row = next(mixed.iterrows())
print(row.dtype, type(row["shares"]))       # object: the ints are no longer an int column

{'shares': dtype('int64'), 'price': dtype('float64'), 'ticker': <StringDtype(na_value=nan)>}
object <class 'int'>


**When a loop is right.** When each step needs the result of the previous step *and* no
built-in expresses it. Some path-dependent things do have a built-in: a running
high-water mark looks like a loop but is `cummax`:

In [17]:
hwm_loop, peak = [], -np.inf
for p in spy:
    peak = max(peak, p)
    hwm_loop.append(peak)
assert np.allclose(hwm_loop, spy.cummax())
print("cummax matches the loop")

cummax matches the loop


A **rebalanced portfolio** does not. Weights drift with each day's returns and are reset
only on rebalance days, so day t's weights depend on day t-1's in a way no single
built-in captures. That's why the backtest engine you rebuild in week 2 loops over
dates. When it does, loop over **numpy arrays** (`rets.to_numpy()`), not `.iloc`, and
put the labels back at the end.

## 5. Missing data

`NaN` means "no value." Three facts cover most of what you need:

1. **Reductions skip it** (`skipna=True`): `mean`, `std`, `sum`, `count`.
2. **Elementwise operations propagate it**: `NaN + 1` is `NaN`; one `NaN` in a
   `cumprod` makes every later value `NaN`.
3. **`NaN != NaN`**. Never test with `== np.nan`; use `.isna()` / `.notna()`.

In [18]:
x = pd.Series([1.0, np.nan, 3.0])
print(x.mean(), x.sum(), x.count(), len(x))     # skipped
print((1 + x).cumprod().tolist())              # propagated
print((x == np.nan).any(), x.isna().sum())      # the == trap

2.0 4.0 2 3
[2.0, nan, 8.0]
False 1


| Tool | Does | Lookahead-safe? |
|---|---|---|
| `dropna()` | drop rows with any NaN (`how="all"`: only all-NaN rows) | yes |
| `fillna(0)` | replace with a constant | yes, but think: is 0 true? |
| `ffill()` | carry the last known value forward | **yes**: uses only the past |
| `ffill(limit=5)` | same, but at most 5 rows | yes |
| `bfill()` | pull the next known value backward | **no**: that's the future |
| `interpolate()` | draw a line between neighbours | **no**: needs the next point |
| `reindex(idx)` | conform to a new index; new labels become NaN | yes |

The pattern you need in week 1 step 3: **a rate on a different calendar**. DTB3 is quoted
on bond-market days, which aren't exactly stock-market days. Here's a synthetic stand-in
that is quoted weekly in percent:

In [19]:
rf_annual_pct = pd.Series(
    np.linspace(0.05, 2.0, 500),
    index=pd.date_range("2015-01-02", periods=500, freq="W-FRI"),
    name="RF",
)
rf_annual_pct.head(3)

2015-01-02    0.0500
2015-01-09    0.0539
2015-01-16    0.0578
Freq: W-FRI, Name: RF, dtype: float64

In [20]:
on_px = rf_annual_pct.reindex(px.index)           # stock dates; most have no quote
print(on_px.isna().sum(), "NaN after reindex")
filled = on_px.ffill()                            # carry each Friday's quote forward
print(filled.isna().sum(), "NaN after ffill (only the days before the first quote)")
filled.loc["2015-01-01":"2015-01-09"]

2020 NaN after reindex
1 NaN after ffill (only the days before the first quote)


date
2015-01-01       NaN
2015-01-02    0.0500
2015-01-05    0.0500
2015-01-06    0.0500
2015-01-07    0.0500
2015-01-08    0.0500
2015-01-09    0.0539
Name: RF, dtype: float64

`reindex` then `ffill` is the point-in-time-correct way to put a slow series on a fast
calendar: each day sees the most recent quote **already published**. `bfill` would hand
Monday the coming Friday's rate, which is a small lookahead. It's rarely the one that
ruins a backtest, but it's the same kind of bug as the one that does.

**Exercise 4.** Using `rf_annual_pct` (annual, in percent): reindex to `px.index`,
forward-fill, convert to a daily decimal rate with your week-1 `daily_rate_from_annual`,
and compute SPY's daily **excess** return (return minus the daily rate). Answer with the
mean daily excess return x 252, rounded to 6 places. (Check: the first rows are NaN
because there's no quote yet, and pandas skips them in the mean.)

Expected:

```
-0.262921
```

**Exercise 5.** Plant a gap: copy `px[["SPY", "IEF"]]`, set SPY to `NaN` on 2016-03-01
through 2016-03-03 (one `.loc`), then compute simple returns. How many NaN SPY returns
are there, and why is it one more than the number of NaN prices (ignoring row 0)? Answer
with `(nan_prices, nan_returns)` for SPY, both counted over the whole frame.

Expected:

```
3

5
```

The answer to Exercise 5's "why": the day *after* the gap has a price but no yesterday.
Returns always have one more NaN per gap than prices do. When you do meet real gaps,
write down what you expected before you fill them.

## What to remember

- Arithmetic aligns on labels first. Missing labels become `NaN`.
- Frame op Series matches the Series to the **columns**; use `.sub(s, axis=0)` to match
  rows.
- `axis=` names the dimension that disappears.
- Reach for a whole-frame expression first. Loop only for true path dependence, and then
  over numpy arrays.
- `ffill` is causal; `bfill` and `interpolate` are not.

## Reading for this notebook

- McKinney, *Python for Data Analysis* 3e, ch. 5.2 "Essential Functionality" (arithmetic
  and data alignment) and ch. 7.1 "Handling Missing Data": https://wesmckinney.com/book/
- VanderPlas, *Python Data Science Handbook*, "Operating on Data in Pandas" and "Handling
  Missing Data": https://jakevdp.github.io/PythonDataScienceHandbook/
- pandas user guide, "Essential basic functionality" and "Working with missing data":
  https://pandas.pydata.org/docs/user_guide/index.html